In [1]:
from data_loader import get_mix_instruct
from utility_functions.delift_se import get_delift_se_utility
from subset import create_subset, get_subset

prompts, references, ds_name = get_mix_instruct("train", 21000)
utility, utility_name = get_delift_se_utility(prompts, references, ds_name)
subset, subset_name = create_subset(utility, utility_name, k=1)
s_prompts, s_references = get_subset(subset, prompts, references)

Dataset: mix-instruct_train_21000 found in cache, loading from cache ✅
Utility: mix-instruct_train_21000_delift-se found in cache, loading from cache ✅
Subset: mix-instruct_train_21000_delift-se_1 found in cache, loading from cache ✅


In [14]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B')

def formatting_prompts_func(prompts, references):
    return [
        f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

{prompt}

### Response:
{reference}
                """
        for prompt, reference in zip(prompts, references)
    ]

def preprocess_function(examples):
    inputs = tokenizer(examples["text"], truncation=True)
    return inputs

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [16]:
from torch.utils.data import DataLoader
from datasets import Dataset
import numpy as np

def get_data_loaders(prompts, references, subset, bs, formatting_fn, preprocess_fn, data_collator):
    prompts, references = np.array(prompts), np.array(references)
    domain_indices = []
    domains = []
    for s in subset:
        index, value = s[0], s[1]
        if value > 30:
            domain_indices.append([index])
            domains.append(index)
        else:
            d = np.argmax(utility[index, domains])
            domain_indices[d].append(index)

    data_loaders = []
    for idx in domain_indices:
        domain_prompts, domain_references = prompts[idx], references[idx]
        text = formatting_fn(domain_prompts, domain_references)
        ds = Dataset.from_dict({"text": text})
        ds = ds.map(preprocess_fn, batched=True)
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
        dl = DataLoader(ds, batch_size=bs, shuffle=False, collate_fn=data_collator)
        data_loaders.append(dl)
    return data_loaders


data_loaders = get_data_loaders(prompts, references, subset, 12, formatting_prompts_func, preprocess_function, data_collator)

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/857 [00:00<?, ? examples/s]

Map:   0%|          | 0/1454 [00:00<?, ? examples/s]

Map:   0%|          | 0/1303 [00:00<?, ? examples/s]

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Map:   0%|          | 0/1391 [00:00<?, ? examples/s]